# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
"""Two findings from the FlyRank research paper (docs/flyrank-seo-research-march-2026.pdf) that sit right next to my lane. For each: where the label comes from, whether the validation design carries the claim, and what would make it stronger. Then a quick check on the starter slice.
The starter slice is NOT the paper's dataset (32 clients vs 57 brands, and a different trend threshold), so my checks cant refute anything. They only show whether the pattern is sensitive to the things I ask about."""

import sys
sys.path.insert(0, "../scripts")
import numpy as np
import pandas as pd
import lane_features as lf
import lane_baseline as lb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score

raw = lf.load_raw()
df = lf.build_features(raw)
X = lf.feature_matrix(df)
y = df["is_declining_label"].to_numpy()

print("""FINDING #1 (p.6, plus the logistic regression in the ML appendix p.29): "growing content is longer and younger than declining content" (3.2K vs 2.3K words, 184 vs 230 days), and a logistic regression separates growing from declining pages at "71% holdout accuracy".
- label: trend_direction from last 30d vs previous 30d impressions. the paper defines up/down at +-10% (p.5), the starter data dictionary uses +-20%, so the same page can be "down" in one and "stable" in the other.
- validation: the comparison is pooled across 57 brands with no within-brand check, so brand mix could create the whole gap (brands with older, thinner pages that happen to be declining). the model uses an 80/20 split (p.30) with no mention of grouping by brand, and 71% accuracy is reported without a base rate. the paper's own counts are 74.8K up vs 45.6K down, so always guessing "up" already gets ~62%. features like days visible and impressions are 90 day numbers, which overlap the 30 day trend window (my w03 test showed that overlap alone moves AUC from 0.85 to 0.94).
- to make it stronger: report the majority-class rate next to the 71%, split by brand, compare up vs down inside each brand, and use only pre-window features.""")

ud = df[df["trend_direction"].isin(["up", "down"])]
display(ud.groupby("trend_direction").agg(pages=("content_id", "size"), avg_age_days=("content_age_days", "mean"),
                                          avg_words_measured=("word_count", lambda s: s[s > 0].mean())).round(0))
gaps = [g.loc[g["trend_direction"] == "down", "content_age_days"].mean() - g.loc[g["trend_direction"] == "up", "content_age_days"].mean()
        for _, g in ud.groupby("client_id") if (g["trend_direction"] == "up").sum() >= 30 and (g["trend_direction"] == "down").sum() >= 30]
print(f"starter check: declining pages are on average YOUNGER and LONGER here, the opposite of the paper. inside the {len(gaps)} clients with 30+ up and 30+ down pages, "
      f"declining pages are older in {sum(g > 0 for g in gaps)} and younger in {sum(g < 0 for g in gaps)}. so on this slice the age gap has no direction once you look inside clients.")

print("""
FINDING #4 (p.9): "365+ day content that was refreshed within 30 days shows 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4039)", followed by "run a recurring refresh program".
- label/outcome: health score is a FlyRank composite that includes impressions (p.5), and impressions are a 90 day total. "refreshed within 30 days" is measured on export day.
- validation: its a cross-section, not before/after on the same pages, and there is no comparison group of similar pages. two problems. timing: 60 of the 90 days of impressions happened BEFORE a refresh that is less than 30 days old, so the refresh cant have caused most of that gap. selection: someone chose which old pages to refresh, likely the ones that still mattered, so the gap could be the choosing. "x3.2" and "x57" read causal, the paper's own limits section says its observational.
- to make it stronger: compare impressions BEFORE the refresh date between refreshed and not refreshed pages (if the gap is already there, its selection), do it inside each brand, and ideally a before/after with a matched untouched group.""")

old = df[df["content_age_days"] >= 365].assign(refreshed=lambda d: np.where(d["days_since_last_update"] < 30, "refreshed <30d", "not refreshed"))
display(old.groupby("refreshed").agg(pages=("content_id", "size"), clients=("client_id", "nunique"),
                                     median_impr_days61_90_before=("impr_d61_90", "median"), median_impr_90d=("impressions_90d", "median")).round(0))
top_client_share = old.loc[old["refreshed"] == "refreshed <30d", "client_id"].value_counts(normalize=True).iloc[0]
print(f"starter check: here the 'refreshed' old pages have FEWER impressions, already in days 61-90 before any refresh, and {top_client_share:.0%} of them are one client "
      "while the other group has only a few clients. so 'refreshed vs not' is mostly 'which client', exactly the selection problem. no 57x shows up on this slice.")

FINDING #1 (p.6, plus the logistic regression in the ML appendix p.29): "growing content is longer and younger than declining content" (3.2K vs 2.3K words, 184 vs 230 days), and a logistic regression separates growing from declining pages at "71% holdout accuracy".
- label: trend_direction from last 30d vs previous 30d impressions. the paper defines up/down at +-10% (p.5), the starter data dictionary uses +-20%, so the same page can be "down" in one and "stable" in the other.
- validation: the comparison is pooled across 57 brands with no within-brand check, so brand mix could create the whole gap (brands with older, thinner pages that happen to be declining). the model uses an 80/20 split (p.30) with no mention of grouping by brand, and 71% accuracy is reported without a base rate. the paper's own counts are 74.8K up vs 45.6K down, so always guessing "up" already gets ~62%. features like days visible and impressions are 90 day numbers, which overlap the 30 day trend window (my w03 tes

,pages,avg_age_days,avg_words_measured
trend_direction,,,
down,16262,236.0,3222.0
up,4388,288.0,2998.0


starter check: declining pages are on average YOUNGER and LONGER here, the opposite of the paper. inside the 18 clients with 30+ up and 30+ down pages, declining pages are older in 8 and younger in 10. so on this slice the age gap has no direction once you look inside clients.

FINDING #4 (p.9): "365+ day content that was refreshed within 30 days shows 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4039)", followed by "run a recurring refresh program".
- label/outcome: health score is a FlyRank composite that includes impressions (p.5), and impressions are a 90 day total. "refreshed within 30 days" is measured on export day.
- validation: its a cross-section, not before/after on the same pages, and there is no comparison group of similar pages. two problems. timing: 60 of the 90 days of impressions happened BEFORE a refresh that is less than 30 days old, so the refresh cant have caused most of that gap. selection: someone chose which old pages to refresh, likely the o

,pages,clients,median_impr_days61_90_before,median_impr_90d
refreshed,,,,
not refreshed,553,4,519.0,1159.0
refreshed <30d,5807,9,326.0,817.0


starter check: here the 'refreshed' old pages have FEWER impressions, already in days 61-90 before any refresh, and 52% of them are one client while the other group has only a few clients. so 'refreshed vs not' is mostly 'which client', exactly the selection problem. no 57x shows up on this slice.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
"""My week 5 model (logistic regression, same features) under three splits:
- BEFORE: a random 80/20 row split, the design the paper used. pages from the same client end up on both sides.
- the single client holdout from w05 (6 unseen clients).
- AFTER: 5-fold GroupKFold by client, every client is the test set exactly once, I report mean and spread.
The frozen rule baseline is scored on the exact same test rows in every split. Base rate next to every number."""

def logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def metrics(y_true, s):
    return {"AUC": roc_auc_score(y_true, s), "P@50": lf.precision_at_k(y_true, s, 50), "P@200": lf.precision_at_k(y_true, s, 200),
            "accuracy@0.5": float(((s >= 0.5) == y_true).mean()), "base rate": float(y_true.mean())}

base_score = lb.baseline_score(df).to_numpy()
rows = []

# before: random row split
tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=lf.SEED, stratify=y)
s = logreg().fit(X.iloc[tr_idx], y[tr_idx]).predict_proba(X.iloc[te_idx])[:, 1]
rows.append({"split": "random 80/20 rows (before)", "model": "logistic regression", **metrics(y[te_idx], s)})
rows.append({"split": "random 80/20 rows (before)", "model": "rule baseline", **metrics(y[te_idx], base_score[te_idx]), "accuracy@0.5": np.nan})

# w05 client holdout
train, test = lf.client_holdout(df)
s = logreg().fit(X[train], y[train]).predict_proba(X[test])[:, 1]
rows.append({"split": "client holdout, 6 clients (w05)", "model": "logistic regression", **metrics(y[test], s)})
rows.append({"split": "client holdout, 6 clients (w05)", "model": "rule baseline", **metrics(y[test], base_score[test]), "accuracy@0.5": np.nan})

# after: 5-fold grouped by client, also keep out-of-fold scores for section 3
oof = np.zeros(len(df))
folds = []
for k, (a, b) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=df["client_id"]), start=1):
    s = logreg().fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    oof[b] = s
    folds.append({"fold": k, "test clients": df["client_id"].iloc[b].nunique(), "test rows": len(b),
                  **{f"model {m}": v for m, v in metrics(y[b], s).items() if m != "accuracy@0.5"},
                  "rule P@50": lf.precision_at_k(y[b], base_score[b], 50), "rule P@200": lf.precision_at_k(y[b], base_score[b], 200)})
folds = pd.DataFrame(folds).set_index("fold")
display(folds.round(3))

for model_name, cols in [("logistic regression", ["model AUC", "model P@50", "model P@200", "model base rate"]), ("rule baseline", [None, "rule P@50", "rule P@200", "model base rate"])]:
    entry = {"split": "5-fold grouped by client (after)", "model": model_name}
    for metric, col in zip(["AUC", "P@50", "P@200", "base rate"], cols):
        entry[metric] = f"{folds[col].mean():.3f} +- {folds[col].std():.3f}" if col else ""
    rows.append(entry)
summary = pd.DataFrame(rows).set_index(["split", "model"])
for col in ["AUC", "P@50", "P@200", "accuracy@0.5", "base rate"]:
    summary[col] = summary[col].map(lambda v: v if isinstance(v, str) else ("" if pd.isna(v) else f"{v:.3f}"))
display(summary)

wins = int((folds["model P@50"] > folds["rule P@50"]).sum())
print(f"""what changed:
- random split AUC {summary.loc[('random 80/20 rows (before)', 'logistic regression'), 'AUC']} -> grouped {folds['model AUC'].mean():.3f}. the gap is what the model gets for free when it has seen the same clients in training, and its why the paper's 80/20 number is optimistic.
- my w05 holdout (AUC {summary.loc[('client holdout, 6 clients (w05)', 'logistic regression'), 'AUC']}) was an easy fold. across all 5 the AUC swings {folds['model AUC'].min():.2f}-{folds['model AUC'].max():.2f}, the fold with one giant client is the hardest.
- the result that survives: at P@50 the model beats the frozen rule in {wins} of 5 folds ({folds['model P@50'].mean():.2f} vs {folds['rule P@50'].mean():.2f} on average), and at P@200 too ({folds['model P@200'].mean():.2f} vs {folds['rule P@200'].mean():.2f}).
- my w02 target was P@50 >= 0.80. the grouped average is {folds['model P@50'].mean():.2f} +- {folds['model P@50'].std():.2f}, so I hit it on some folds, not reliably. thats the honest number.""")

,test clients,test rows,model AUC,model P@50,model P@200,model base rate,rule P@50,rule P@200
fold,,,,,,,,
1,1,7008,0.601,0.70,0.750,0.490,0.74,0.630
2,7,5731,0.623,0.82,0.850,0.645,0.66,0.705
3,8,5753,0.766,0.66,0.665,0.379,0.50,0.475
4,8,5755,0.726,0.80,0.790,0.622,0.70,0.735
5,8,5753,0.757,0.98,0.910,0.585,0.86,0.920


AUC  \
split                            model                                 
random 80/20 rows (before)       logistic regression           0.737   
                                 rule baseline                 0.664   
client holdout, 6 clients (w05)  logistic regression           0.844   
                                 rule baseline                 0.804   
5-fold grouped by client (after) logistic regression  0.695 +- 0.077   
                                 rule baseline                         

                                                                P@50  \
split                            model                                 
random 80/20 rows (before)       logistic regression           0.880   
                                 rule baseline                 0.760   
client holdout, 6 clients (w05)  logistic regression           0.820   
                                 rule baseline                 0.720   
5-fold grouped by client (after) logistic regression  0.792 +- 0.125   
                                 rule baseline        0.692 +- 0.131   

                                                               P@200  \
split                            model                                 
random 80/20 rows (before)       logistic regression           0.855   
                                 rule baseline                 0.670   
client holdout, 6 clients (w05)  logistic regression           0.790   
                                 rule baseline                 0.670   
5-fold grouped by client (after) logistic regression  0.793 +- 0.094   
                                 rule baseline        0.693 +- 0.162   

                                                     accuracy@0.5  \
split                            model                              
random 80/20 rows (before)       logistic regression        0.669   
                                 rule baseline                      
client holdout, 6 clients (w05)  logistic regression        0.745   
                                 rule baseline                      
5-fold grouped by client (after) logistic regression                
                                 rule baseline                      

                                                           base rate  
split                            model                                
random 80/20 rows (before)       logistic regression           0.542  
                                 rule baseline                 0.542  
client holdout, 6 clients (w05)  logistic regression           0.391  
                                 rule baseline                 0.391  
5-fold grouped by client (after) logistic regression  0.544 +- 0.110  
                                 rule baseline        0.544 +- 0.110

what changed:
- random split AUC 0.737 -> grouped 0.695. the gap is what the model gets for free when it has seen the same clients in training, and its why the paper's 80/20 number is optimistic.
- my w05 holdout (AUC 0.844) was an easy fold. across all 5 the AUC swings 0.60-0.77, the fold with one giant client is the hardest.
- the result that survives: at P@50 the model beats the frozen rule in 4 of 5 folds (0.79 vs 0.69 on average), and at P@200 too (0.79 vs 0.69).
- my w02 target was P@50 >= 0.80. the grouped average is 0.79 +- 0.12, so I hit it on some folds, not reliably. thats the honest number.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
"""The w03 hunt again, now on the final model and out-of-fold (5-fold grouped by client), with the skill's checklist ticked off by code, not by me saying so."""

def oof_auc(features, shuffle_seed=None):
    # scored per fold and averaged, same as section 2. pooling the out-of-fold scores first would mix 5 different
    # base rates, which drags a no-skill model below 0.5 (I tripped on that the first time)
    aucs, p50s = [], []
    for a, b in GroupKFold(n_splits=5).split(features, y, groups=df["client_id"]):
        y_fit = y[a] if shuffle_seed is None else np.random.default_rng(shuffle_seed).permutation(y[a])
        s = logreg().fit(features.iloc[a], y_fit).predict_proba(features.iloc[b])[:, 1]
        aucs.append(roc_auc_score(y[b], s))
        p50s.append(lf.precision_at_k(y[b], s, 50))
    return np.mean(aucs), np.mean(p50s)

final_auc, final_p50 = folds["model AUC"].mean(), folds["model P@50"].mean()
in_sample = logreg().fit(X, y).predict_proba(X)[:, 1]

checks = {}
# 1 timeline: map every model column to the raw columns it is built from, all of them must be known before day -30
sources = {
    "search_volume": ["search_volume"], "competition": ["competition"], "cpc": ["cpc"], "has_keyword_data": ["search_volume"],
    "word_count": ["word_count"], "has_word_count": ["word_count"], "age_at_prediction": ["content_age_days"],
    "log_impr_prev30": ["impressions_prev_30d"], "log_clicks_prev30": ["clicks_prev_30d"], "log_sessions_prev30": ["sessions_prev_30d"],
    "log_impr_d61_90": ["days 61-90"], "log_clicks_d61_90": ["days 61-90"], "log_sessions_d61_90": ["days 61-90"],
    "impr_momentum_prior": ["impressions_prev_30d", "days 61-90"], "ctr_prev30": ["clicks_prev_30d", "impressions_prev_30d"],
    "prev30_empty": ["impressions_prev_30d"],
}
for cat in lf.CATEGORICAL_FEATURES:
    sources.update({c: [cat] for c in X.columns if c.startswith(cat + "_")})
before_day_minus_30 = {"search_volume", "competition", "cpc", "word_count", "content_age_days", "impressions_prev_30d", "clicks_prev_30d",
                       "sessions_prev_30d", "days 61-90", "competition_level", "content_type", "main_intent"}
# ("days 61-90" = 90d - last_30d - prev_30d, the only place the label window is touched, w03 test 2 + w04 check 1)
unmapped = [c for c in X.columns if c not in sources]
late = [c for c, src in sources.items() if not set(src) <= before_day_minus_30]
checks[f"timeline: all {X.shape[1]} columns built only from things known before day -30 (unmapped {unmapped}, late {late})"] = not unmapped and not late
# 2 label-derived / sibling columns absent
checks["no trend_direction / trend_pct / last_30d column in X"] = not any(k in c for c in X.columns for k in ["trend", "last_30d", "last30"])
# 3 product flags
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"]
checks["no product flags / scores (none exist in the file either)"] = not any(f in c for c in X.columns for f in product_flags) and not any(f in raw.columns for f in product_flags)
# 4 ids
checks["no id columns"] = not any(c.endswith("_id") for c in X.columns)
# 5 grouped split
checks["split grouped by client (no client on both sides)"] = all(
    set(df["client_id"].iloc[a]).isdisjoint(df["client_id"].iloc[b]) for a, b in GroupKFold(n_splits=5).split(X, y, groups=df["client_id"]))
for name, ok in checks.items():
    print(("[x] " if ok else "[ ] ") + name)

# 6 harness + attack tests, all out-of-fold
attack = {
    "final model (out-of-fold, mean of 5 folds)": (final_auc, final_p50),
    "+ trend_pct (harness check, should jump to ~1)": oof_auc(X.assign(trend_pct=raw["trend_pct"].fillna(0))),
    "+ raw impressions_90d (window arithmetic)": oof_auc(X.assign(impressions_90d=np.log1p(raw["impressions_90d"]))),
    "without prev30_empty (top feature, train-without)": oof_auc(X.drop(columns=["prev30_empty"])),
    # one shuffle is noisy (a random weight on a strong feature can land far from 0.5 either way), so 5 shuffles averaged
    "shuffled training labels (control, mean of 5 shuffles)": tuple(np.mean([oof_auc(X, shuffle_seed=s) for s in range(5)], axis=0)),
}
attack_table = pd.DataFrame(attack, index=["AUC", "P@50"]).T.round(3)
attack_table["base rate"] = round(y.mean(), 3)
display(attack_table)
print(f"[x] base rate printed next to every metric ({y.mean():.3f})")
print(f"[x] metrics out-of-fold, not in-sample: in-sample AUC {roc_auc_score(y, in_sample):.3f} vs out-of-fold {final_auc:.3f} (pooled out-of-fold scores: {roc_auc_score(y, oof):.3f})")
print(f"""
reading it: the harness still catches a planted leak (trend_pct -> {attack['+ trend_pct (harness check, should jump to ~1)'][0]:.3f}) and the 90d subtraction leak is still there if I let it in ({attack['+ raw impressions_90d (window arithmetic)'][0]:.3f}).
dropping my top feature moves AUC {final_auc:.3f} -> {attack['without prev30_empty (top feature, train-without)'][0]:.3f}, no collapse at all (log_impr_prev30 = 0 says the same thing), so no single column is carrying a hidden answer. P@50 even goes up without it ({attack['without prev30_empty (top feature, train-without)'][1]:.2f} vs {final_p50:.2f}), but thats inside the fold-to-fold spread (+-{folds['model P@50'].std():.2f}) and picking features by looking at test folds would be tuning on the test set, so the feature set stays frozen. shuffled labels give {attack['shuffled training labels (control, mean of 5 shuffles)'][0]:.3f}.
the one judgement call left is word_count (measured at export, could postdate day -30 for edited pages), its a weak feature so the risk is small, but its noted.""")

[x] timeline: all 28 columns built only from things known before day -30 (unmapped [], late [])
[x] no trend_direction / trend_pct / last_30d column in X
[x] no product flags / scores (none exist in the file either)
[x] no id columns
[x] split grouped by client (no client on both sides)


,AUC,P@50,base rate
"final model (out-of-fold, mean of 5 folds)",0.695,0.792,0.542
"+ trend_pct (harness check, should jump to ~1)",0.999,1.000,0.542
+ raw impressions_90d (window arithmetic),0.892,0.992,0.542
"without prev30_empty (top feature, train-without)",0.696,0.848,0.542
"shuffled training labels (control, mean of 5 shuffles)",0.495,0.464,0.542


[x] base rate printed next to every metric (0.542)
[x] metrics out-of-fold, not in-sample: in-sample AUC 0.748 vs out-of-fold 0.695 (pooled out-of-fold scores: 0.697)

reading it: the harness still catches a planted leak (trend_pct -> 0.999) and the 90d subtraction leak is still there if I let it in (0.892).
dropping my top feature moves AUC 0.695 -> 0.696, no collapse at all (log_impr_prev30 = 0 says the same thing), so no single column is carrying a hidden answer. P@50 even goes up without it (0.85 vs 0.79), but thats inside the fold-to-fold spread (+-0.12) and picking features by looking at test folds would be tuning on the test set, so the feature set stays frozen. shuffled labels give 0.495.
the one judgement call left is word_count (measured at export, could postdate day -30 for edited pages), its a weak feature so the risk is small, but its noted.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
"""My boldest sentence so far was in w01: "We will be able to successfully predict page relevance and performance over trend relevant timeframes and even provide editors with actionable data for example a usually well performing item is predicted to decline over the next 30 days so editors can pre-emptively work on refreshing it..."
And the w02 one: "Fixed rules decay quickly at scale; ML holds precision."
Rewriting both with the claim ladder: observed / measured / ranks at P@K / decision-support, numbers with their base rate, and no "will" or "successfully predict"."""

p50_m, p50_s = folds["model P@50"].mean(), folds["model P@50"].std()
rule_p50 = folds["rule P@50"].mean()
print(f"""BEFORE (w01): "We will be able to successfully predict page relevance and performance over trend relevant timeframes..."

AFTER: "On one 90-day snapshot of 32 pseudonymized clients, a logistic regression that only uses data from before the last 30 days ranked pages that went on to lose 20%+ of their search impressions. Across 5 client-grouped folds, {p50_m:.0%} (+-{p50_s:.0%}) of its top 50 picks per fold declined, against a {y.mean():.0%} base rate and {rule_p50:.0%} for a transparent rule. We read this as decision-support for which pages to review first, not as a forecast of Google rankings or evidence that refreshing a page prevents the drop."

BEFORE (w02): "Fixed rules decay quickly at scale; ML holds precision."

AFTER: "In this data a learned model kept a higher hit rate than my rule at the top of the list (P@50 {p50_m:.2f} vs {rule_p50:.2f}) and deeper into it (P@200 {folds['model P@200'].mean():.2f} vs {folds['rule P@200'].mean():.2f}), averaged over 5 client folds. The gap varies a lot by fold and the rule is better at P@50 in {5 - wins} of the 5, so the rule stays as the readable fallback."

what changed and why:
- "will be able to successfully predict" -> "ranked ... at P@50 of": the evidence is an out-of-sample ranking score on one snapshot, not a forecast that was checked in a later month.
- "performance / relevance" -> "lost 20%+ of search impressions": thats what the label actually is, a GSC impressions threshold, not relevance.
- added the base rate, the rule baseline and the spread across folds, a number without those means nothing.
- "editors can pre-emptively refresh it" -> "which pages to review first": nothing here tests whether refreshing helps. that would need a before/after with an untouched comparison group.
- "ML holds precision" -> a measured comparison with its spread, because one fold goes the other way.""")

BEFORE (w01): "We will be able to successfully predict page relevance and performance over trend relevant timeframes..."

AFTER: "On one 90-day snapshot of 32 pseudonymized clients, a logistic regression that only uses data from before the last 30 days ranked pages that went on to lose 20%+ of their search impressions. Across 5 client-grouped folds, 79% (+-12%) of its top 50 picks per fold declined, against a 54% base rate and 69% for a transparent rule. We read this as decision-support for which pages to review first, not as a forecast of Google rankings or evidence that refreshing a page prevents the drop."

BEFORE (w02): "Fixed rules decay quickly at scale; ML holds precision."

AFTER: "In this data a learned model kept a higher hit rate than my rule at the top of the list (P@50 0.79 vs 0.69) and deeper into it (P@200 0.79 vs 0.69), averaged over 5 client folds. The gap varies a lot by fold and the rule is better at P@50 in 1 of the 5, so the rule stays as the readable fallback."

w

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.